In [ ]:
import sys, json, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter

from src.utils import DATASETS, make_splits
from src.fsa import predict_log_time, fit_sigma, survival_lognormal, predicted_median

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = sns.color_palette('tab10')
os.makedirs('../figures', exist_ok=True)

DATASET = 'whas500'  # change to explore others

## 1. Censoring Summary

In [ ]:
rows = []
for name, loader in DATASETS.items():
    X, T, Delta = loader()
    rows.append({
        'dataset':         name,
        'n':               len(T),
        'censoring_rate':  f'{(1 - Delta.mean()):.1%}',
        'median_followup': f'{np.median(T):.1f}',
        'time_range':      f'{T.min():.1f} – {T.max():.1f}',
    })
pd.DataFrame(rows).set_index('dataset')

## 2. Survival Curves vs KM (by Risk Quartile)

In [ ]:
X, T, Delta = DATASETS[DATASET]()
tr_idx, te_idx = make_splits(len(T), n_splits=1)[0]
X_tr, T_tr, D_tr = X[tr_idx], T[tr_idx], Delta[tr_idx]
X_te, T_te, D_te = X[te_idx], T[te_idx], Delta[te_idx]

uncensored = D_tr == 1
X_all  = np.vstack([X_tr, X_te])
mu_all = predict_log_time(X_tr[uncensored], T_tr[uncensored], X_all)  # actual T, not log T
mu_tr, mu_te = mu_all[:len(X_tr)], mu_all[len(X_tr):]
sigma = fit_sigma(T_tr, D_tr, mu_tr)
print(f'σ = {sigma:.3f}')

t_grid = np.linspace(np.percentile(T_tr, 1), np.percentile(T_tr, 99), 300)
S   = survival_lognormal(t_grid, mu_te, sigma)
med = predicted_median(S, t_grid)

In [ ]:
# Stratify test subjects into risk quartiles by predicted median survival
finite_med = med[np.isfinite(med)]
quartile_cuts = np.percentile(finite_med, [25, 50, 75])
bins   = np.digitize(med, quartile_cuts)   # 0 = highest risk, 3 = lowest risk
labels = ['Q1 (high risk)', 'Q2', 'Q3', 'Q4 (low risk)']

fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=True)
kmf = KaplanMeierFitter()

for q, (ax, label) in enumerate(zip(axes, labels)):
    mask = bins == q
    if mask.sum() == 0:
        ax.set_visible(False)
        continue
    ax.plot(t_grid, S[mask].mean(0), color=PALETTE[q], lw=2, label='Predicted (mean)')
    kmf.fit(T_te[mask], event_observed=D_te[mask])
    kmf.plot_survival_function(ax=ax, ci_show=True, color=PALETTE[q],
                               linestyle='--', linewidth=1.5, label='KM')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Time')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_ylabel('S(t)')
fig.suptitle(f'Predicted survival vs KM — {DATASET}  (σ={sigma:.2f})', fontsize=13)
plt.tight_layout()
plt.savefig(f'../figures/{DATASET}_km_quartiles.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'../figures/{DATASET}_km_quartiles.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Individual Survival Curves

In [ ]:
# 6 subjects evenly spread across the risk spectrum (by predicted median)
order  = np.argsort(med)
n      = len(order)
picks  = order[[0, n//5, 2*n//5, 3*n//5, 4*n//5, -1]]

fig, ax = plt.subplots(figsize=(8, 5))
for i, j in enumerate(picks):
    m = med[j]
    lbl = f'Subject {j}  (med={m:.0f})' if np.isfinite(m) else f'Subject {j}  (med>grid)'
    ax.plot(t_grid, S[j], color=PALETTE[i], lw=2, label=lbl)

ax.axhline(0.5, color='grey', linestyle=':', lw=1, label='S=0.5')
ax.set_xlabel('Time')
ax.set_ylabel('S(t | x)')
ax.set_ylim(0, 1.05)
ax.set_title(f'Individual survival curves — {DATASET}')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'../figures/{DATASET}_individual_curves.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'../figures/{DATASET}_individual_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. σ Stability (from saved results)

In [ ]:
with open('../results/results.json') as f:
    store = json.load(f)

sigma_results = {
    ds: [s['sigma'] for s in splits['fsa']]
    for ds, splits in store['datasets'].items()
}

fig, ax = plt.subplots(figsize=(7, 4))
bp = ax.boxplot(
    sigma_results.values(),
    labels=sigma_results.keys(),
    patch_artist=True,
    medianprops=dict(color='black', lw=1.5),
    boxprops=dict(facecolor='steelblue', alpha=0.6),
)
ax.axhline(1.0, color='tomato', linestyle='--', lw=1.2, label='σ = 1')
ax.set_ylabel('σ', fontsize=12)
ax.set_title('Fitted σ across datasets and splits', fontsize=12)
ax.tick_params(axis='x', labelsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../figures/sigma_stability.pdf', dpi=300, bbox_inches='tight')
plt.savefig('../figures/sigma_stability.png', dpi=300, bbox_inches='tight')
plt.show()